# Semana 1: Estructuras de Datos para Búsqueda en Bases de Datos (SQL y NoSQL)
## Notebook de Ejercicios (NB2) – Explorando Motores Reales sin Instalación

### Objetivo
Ver cómo la teoría de estructuras de datos (B-Tree, Hash, Listas, LSM-Tree, Grafos) se materializa en motores de bases de datos reales utilizando herramientas online gratuitas y conexiones desde Colab.

### Herramientas y servicios utilizados
*   **SQLiteOnline**: Editor SQL en navegador (sin instalación).
*   **Redis Cloud**: Base de datos en memoria gratuita (30 MB).
*   **MongoDB Atlas**: Base de datos documental gratuita (512 MB).
*   **Astra DB (DataStax)**: Cassandra como servicio gratuito.
*   **Neo4j Sandbox**: Base de datos de grafos con datasets precargados.
*   **Google Colab**: Para conectarnos a estos servicios mediante Python.

### Instrucciones generales
1.  Para cada servicio, sigue los enlaces de registro y crea una cuenta gratuita.
2.  Vuelve a Colab y ejecuta las celdas correspondientes para conectarte y experimentar.
3.  Anota tus observaciones sobre rendimiento y uso de índices.

## Configuración Inicial

Instalamos las librerías necesarias para conectarnos a los diferentes motores.

In [ ]:
# Instalación de librerías necesarias (ejecutar una sola vez)
!pip install redis pymongo cassandra-driver neo4j

print("✅ Librerías instaladas.")

---
## Actividad 1: SQL (B-Tree en acción) con SQLiteOnline

**Herramienta**: [SQLiteOnline](https://sqliteonline.com/)

En esta actividad usaremos un navegador web para experimentar con índices B-Tree en SQLite. No necesitamos código Python aquí.

### Pasos a seguir:

1.  Abre [SQLiteOnline](https://sqliteonline.com/).
2.  En el panel izquierdo, asegúrate de que está seleccionado "SQLite".
3.  Ejecuta el siguiente script para crear una tabla con 10,000 registros:

    ```sql
    -- Crear tabla
    CREATE TABLE datos (
        id INTEGER PRIMARY KEY,
        valor INTEGER,
        texto TEXT
    );

    -- Insertar 10,000 registros con valores aleatorios
    WITH RECURSIVE
        cnt(x) AS (VALUES(1) UNION ALL SELECT x+1 FROM cnt WHERE x<10000)
    INSERT INTO datos (id, valor, texto)
    SELECT x, ABS(RANDOM()) % 10000, 'Texto_' || x FROM cnt;
    ```

4.  **Búsqueda sin índice**: Ejecuta una consulta de búsqueda por el campo `valor` (que no tiene índice):

    ```sql
    SELECT * FROM datos WHERE valor = 5000;
    ```

    Anota el tiempo aproximado (lo muestra la interfaz).

5.  **Verificar el plan de ejecución**: Usa `EXPLAIN QUERY PLAN` para confirmar que se está haciendo un table scan:

    ```sql
    EXPLAIN QUERY PLAN SELECT * FROM datos WHERE valor = 5000;
    ```

    Deberías ver "SCAN datos".

6.  **Crear un índice B-Tree**:

    ```sql
    CREATE INDEX idx_valor ON datos(valor);
    ```

7.  **Búsqueda con índice**: Repite la consulta y el `EXPLAIN QUERY PLAN`:

    ```sql
    SELECT * FROM datos WHERE valor = 5000;
    EXPLAIN QUERY PLAN SELECT * FROM datos WHERE valor = 5000;
    ```

    Ahora deberías ver "SEARCH datos USING INDEX idx_valor". El tiempo debería ser mucho menor.

8.  **Reflexión**: ¿Cuánto más rápido fue con índice? ¿Por qué?

---
## Actividad 2: Redis (Estructuras en Memoria) con Redis Cloud

**Servicio**: [Redis Cloud](https://redis.com/try-free/) - 30 MB gratis.

### 2.1. Crear una base de datos Redis

1.  Regístrate en [Redis Cloud](https://redis.com/try-free/).
2.  Crea una nueva base de datos gratuita (Fixed-size free).
3.  Una vez creada, anota los siguientes datos de conexión:
    *   **Endpoint**: `redis-xxxx.region.cloud.redislabs.com:port`
    *   **Contraseña** (password)

### 2.2. Conectar desde Colab y experimentar

In [ ]:
import redis
import time

# Configuración de conexión - ¡REEMPLAZA CON TUS DATOS!
REDIS_HOST = "redis-xxxx.region.cloud.redislabs.com"  # Ej: redis-12345.c1.us-east1-2.gce.cloud.redislabs.com
REDIS_PORT = 12345  # Ej: 12345
REDIS_PASSWORD = "tu_contraseña"

try:
    r = redis.Redis(
        host=REDIS_HOST,
        port=REDIS_PORT,
        password=REDIS_PASSWORD,
        decode_responses=True
    )
    r.ping()  # Verificar conexión
    print("✅ Conectado a Redis Cloud")
except Exception as e:
    print(f"❌ Error de conexión: {e}")
    print("Por favor, verifica tus credenciales y que la base de datos esté activa.")

In [ ]:
# Strings (SET/GET)
r.set("nombre", "Juan Pérez")
r.set("edad", 30)
print(f"GET nombre: {r.get('nombre')}")
print(f"GET edad: {r.get('edad')}")

In [ ]:
# Listas (simulando una cola FIFO)
# Encolar tareas (al final)
r.rpush("cola_tareas", "tarea1", "tarea2", "tarea3")
print(f"Lista cola_tareas: {r.lrange('cola_tareas', 0, -1)}")

# Desencolar (por el principio)
tarea = r.lpop("cola_tareas")
print(f"Tarea procesada: {tarea}")
print(f"Lista restante: {r.lrange('cola_tareas', 0, -1)}")

In [ ]:
# Hashes (objetos)
r.hset("usuario:1001", mapping={"nombre": "Ana", "email": "ana@email.com", "edad": 25})
print(f"HGET nombre: {r.hget('usuario:1001', 'nombre')}")
print(f"HGETALL: {r.hgetall('usuario:1001')}")

In [ ]:
# Sorted Sets (Skip Lists)
r.zadd("ranking", {"jugador1": 100, "jugador2": 200, "jugador3": 150})
print(f"ZRANGE (0 -1): {r.zrange('ranking', 0, -1, withscores=True)}")
print(f"ZRANGE by score (100-180): {r.zrangeby_score('ranking', 100, 180)}")
print(f"ZSCORE jugador2: {r.zscore('ranking', 'jugador2')}")

---
## Actividad 3: MongoDB (Índices y Tipos de Datos) con MongoDB Atlas

**Servicio**: [MongoDB Atlas](https://www.mongodb.com/atlas) - 512 MB gratis.

### 3.1. Crear un cluster gratuito

1.  Regístrate en MongoDB Atlas.
2.  Crea un cluster gratuito (M0).
3.  En "Database Access", crea un usuario y contraseña.
4.  En "Network Access", añade tu IP actual (o 0.0.0.0/0 para permitir cualquier IP, pero no es recomendado).
5.  Obtén la **cadena de conexión** (URI) desde "Connect" > "Connect your application". Debe ser algo como: `mongodb+srv://<usuario>:<contraseña>@cluster0.xxxxx.mongodb.net/`

### 3.2. Conectar desde Colab y experimentar

In [ ]:
from pymongo import MongoClient

# Configuración de conexión - ¡REEMPLAZA CON TU URI!
MONGO_URI = "mongodb+srv://<usuario>:<contraseña>@cluster0.xxxxx.mongodb.net/"

try:
    client = MongoClient(MONGO_URI)
    db = client['test_db']  # base de datos
    collection = db['test_collection']  # colección
    client.admin.command('ping')
    print("✅ Conectado a MongoDB Atlas")
except Exception as e:
    print(f"❌ Error de conexión: {e}")
    print("Verifica tu URI, usuario, contraseña y Network Access.")

In [ ]:
# Insertar documentos con diferentes tipos
docs = [
    {"nombre": "Producto A", "precio": 100, "tags": ["electrónica", "hogar"], "ubicacion": {"type": "Point", "coordinates": [-73.97, 40.77]}},
    {"nombre": "Producto B", "precio": 200, "tags": ["ropa", "hombre"], "ubicacion": {"type": "Point", "coordinates": [-74.00, 40.75]}},
    {"nombre": "Producto C", "precio": 150, "tags": ["electrónica", "computadoras"], "ubicacion": {"type": "Point", "coordinates": [-73.95, 40.78]}},
]

collection.insert_many(docs)
print(f"✅ Insertados {collection.count_documents({})} documentos.")

In [ ]:
# 1. Índice simple (por precio)
collection.create_index("precio")

# Buscar por precio
result = collection.find({"precio": 200}).explain()
print("Plan de consulta para precio=200:")
print(f"  Etapa: {result['queryPlanner']['winningPlan']['stage']}")  # Debería ser IXSCAN si usó índice

# 2. Índice compuesto (por nombre y precio)
collection.create_index([("nombre", 1), ("precio", -1)])

# 3. Índice multikey (sobre array tags)
collection.create_index("tags")

# Buscar por tag
result_tags = collection.find({"tags": "electrónica"}).explain()
print(f"\nBúsqueda por tag 'electrónica' usa índice: {result_tags['queryPlanner']['winningPlan']['stage']}")

# 4. Índice geoespacial
collection.create_index([("ubicacion", "2dsphere")])

# Búsqueda cercana a un punto
cerca = collection.find({
    "ubicacion": {
        "$near": {
            "$geometry": {"type": "Point", "coordinates": [-73.98, 40.76]},
            "$maxDistance": 1000
        }
    }
})
print("\nProductos cercanos:")
for p in cerca:
    print(f"  {p['nombre']}")

---
## Actividad 4: Cassandra (LSM-Tree) con Astra DB (DataStax)

**Servicio**: [Astra DB](https://www.datastax.com/products/datastax-astra) - Gratuito.

### 4.1. Crear una base de datos

1.  Regístrate en Astra DB.
2.  Crea una nueva base de datos (serverless). Elige "Apache Cassandra".
3.  Una vez creada, ve a "Connect" y obtén:
    *   **Secure Connect Bundle** (archivo ZIP).
    *   **Client ID** y **Client Secret** (o token).
4.  Sube el archivo Secure Connect Bundle a Colab (puedes usar el panel de archivos de Colab).

### 4.2. Conectar desde Colab y experimentar

In [ ]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
import time

# Configuración de conexión - ¡REEMPLAZA CON TUS DATOS!
ASTRA_CLIENT_ID = "tu_client_id"
ASTRA_CLIENT_SECRET = "tu_client_secret"
ASTRA_SECURE_BUNDLE_PATH = "/content/secure-connect-tu-base.zip"  # Ruta al archivo subido

try:
    cloud_config = {'secure_connect_bundle': ASTRA_SECURE_BUNDLE_PATH}
    auth_provider = PlainTextAuthProvider(ASTRA_CLIENT_ID, ASTRA_CLIENT_SECRET)
    cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
    session = cluster.connect()
    print("✅ Conectado a Astra DB (Cassandra)")
except Exception as e:
    print(f"❌ Error de conexión: {e}")
    print("Verifica los credenciales y la ruta del bundle.")

In [ ]:
# Crear un keyspace y tabla
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS test_keyspace
    WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}
""")
session.set_keyspace('test_keyspace')

session.execute("""
    CREATE TABLE IF NOT EXISTS usuarios (
        id UUID PRIMARY KEY,
        nombre TEXT,
        email TEXT,
        registro TIMESTAMP
    )
""")
print("✅ Tabla 'usuarios' creada/listo.")

In [ ]:
# Insertar miles de registros y medir velocidad de escritura
from uuid import uuid4
from datetime import datetime

n_inserts = 5000
start = time.time()

for i in range(n_inserts):
    user_id = uuid4()
    nombre = f"usuario_{i}"
    email = f"{nombre}@email.com"
    fecha = datetime.now()
    session.execute(
        "INSERT INTO usuarios (id, nombre, email, registro) VALUES (%s, %s, %s, %s)",
        (user_id, nombre, email, fecha)
    )

elapsed = time.time() - start
print(f"✅ Insertados {n_inserts} registros en {elapsed:.2f} segundos.")
print(f"   Velocidad: {n_inserts/elapsed:.0f} inserciones/segundo.")
print("   (LSM-Trees optimizan escrituras secuenciales)")

In [ ]:
# Realizar una consulta (por clave primaria, eficiente)
rows = session.execute("SELECT * FROM usuarios LIMIT 5")
print("Primeros 5 usuarios:")
for row in rows:
    print(f"  {row.id} - {row.nombre} - {row.email}")

# Discusión: Cassandra usa bloom filters para evitar leer SSTables innecesarias.
# En una lectura por clave (id), el bloom filter confirma rápidamente si el dato existe.
print("\n🔍 En Cassandra, los Bloom Filters evitan revisar SSTables donde seguro no está la clave.")

---
## Actividad 5: Neo4j (Grafos y Relaciones) con Neo4j Sandbox

**Servicio**: [Neo4j Sandbox](https://sandbox.neo4j.com/) - Gratuito, con datasets precargados.

### 5.1. Crear un sandbox

1.  Regístrate en [Neo4j Sandbox](https://sandbox.neo4j.com/).
2.  Crea una nueva instancia. Elige el dataset **"Movies"** (recomendado).
3.  Una vez creada, verás los detalles de conexión:
    *   **URI**: `bolt://xxxx.databases.neo4j.io:7687`
    *   **Usuario**: `neo4j`
    *   **Contraseña**: La proporcionada.

### 5.2. Conectar desde Colab (opcional) y explorar en la interfaz web

Neo4j Sandbox proporciona una interfaz web (Neo4j Browser) muy completa. Puedes ejecutar consultas directamente allí. También podemos conectarnos desde Colab con `neo4j` driver.

In [ ]:
from neo4j import GraphDatabase

# Configuración de conexión - ¡REEMPLAZA CON TUS DATOS!
NEO4J_URI = "bolt://xxxx.databases.neo4j.io:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "tu_contraseña"

class Neo4jConnection:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
    
    def close(self):
        self.driver.close()
    
    def query(self, query, parameters=None):
        with self.driver.session() as session:
            result = session.run(query, parameters)
            return [record for record in result]

try:
    conn = Neo4jConnection(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    conn.driver.verify_connectivity()
    print("✅ Conectado a Neo4j Sandbox")
except Exception as e:
    print(f"❌ Error de conexión: {e}")
    print("Puedes ejecutar las consultas directamente en la interfaz web de Neo4j Sandbox.")

In [ ]:
# Ejemplo de consulta: Actores que trabajaron con un director específico (por ejemplo, 'Tom Hanks')
# Nota: En el dataset Movies, las relaciones son del tipo (:Person)-[:ACTED_IN]->(:Movie) y (:Person)-[:DIRECTED]->(:Movie)

query_actors_with_director = """
MATCH (director:Person)-[:DIRECTED]->(movie:Movie)<-[:ACTED_IN]-(actor:Person)
WHERE director.name = 'Tom Hanks'
RETURN DISTINCT actor.name AS Actor, movie.title AS Movie
"""

try:
    result = conn.query(query_actors_with_director)
    print("Actores que trabajaron con Tom Hanks:")
    for record in result[:10]:  # mostrar primeros 10
        print(f"  {record['Actor']} en {record['Movie']}")
except:
    print("Ejecuta esta consulta en la interfaz web de Neo4j Sandbox.")

In [ ]:
# Camino más corto entre dos actores (por ejemplo, 'Tom Hanks' y 'Meg Ryan')
query_shortest_path = """
MATCH p=shortestPath(
    (a:Person {name: 'Tom Hanks'})-[:ACTED_IN|DIRECTED*]-(b:Person {name: 'Meg Ryan'})
)
RETURN p
"""

print("Consulta de camino más corto:")
print(query_shortest_path)
print("\n👉 En la interfaz web de Neo4j, puedes visualizar el grafo resultante.")

---
## Ejercicios para el Estudiante

1.  **SQLiteOnline**: Prueba a buscar por `texto` en la tabla `datos`. Crea un índice en `texto` y compara tiempos.

2.  **Redis**: Implementa una cola de mensajes simple usando listas de Redis. Un proceso escribe (productor) y otro lee (consumidor) en un bucle.

3.  **MongoDB**: Crea un índice geoespacial en una colección de lugares (puedes generar puntos aleatorios) y realiza una búsqueda de cercanía.

4.  **Cassandra**: Inserta 10,000 registros adicionales y mide el tiempo. ¿La velocidad de inserción se mantiene constante? (Relaciona con escrituras secuenciales de LSM-Tree).

5.  **Neo4j**: En el dataset Movies, encuentra todas las películas en las que actuó un actor y un director específico (por ejemplo, 'Keanu Reeves' y 'Lana Wachowski').

6.  **Reflexión final**: ¿Qué estructura de datos viste en cada motor? Completa la tabla resumen del NB1 con las observaciones de esta práctica.

---
## Conclusiones de la Sesión

*   Hemos conectado con **motores reales** utilizando servicios cloud gratuitos, demostrando que no es necesario instalar software localmente.
*   En **SQLite** confirmamos experimentalmente la mejora de rendimiento de los índices B-Tree (de O(n) a O(log n)).
*   En **Redis** exploramos estructuras en memoria: Strings (hash), Lists (listas enlazadas), Hashes y Sorted Sets (Skip Lists).
*   En **MongoDB** creamos diferentes tipos de índices (simple, compuesto, multikey, geoespacial) y verificamos su uso con `explain()`.
*   En **Cassandra** medimos la alta velocidad de escritura característica de los LSM-Trees y discutimos el rol de los Bloom Filters.
*   En **Neo4j** exploramos un grafo de películas, ejecutando consultas de relaciones y caminos.

¡Ahora has visto la teoría de estructuras de datos aplicada en motores reales!